In [1]:
import zipfile
import xml.etree.ElementTree as ET
import os
from pathlib import Path
import shutil

# Set up directories
INPUT_DIR = "./kmz_files_final"  
OUTPUT_DIR = "./kmz_files_edited"  
NAMESPACES = {
    'kml': 'http://www.opengis.net/kml/2.2'
}

def remove_polygons_from_folder(folder_elem):
    """Remove first and last Polygon elements from a Folder"""
    ns = '{http://www.opengis.net/kml/2.2}'
    
    # Get all Placemarks that contain Polygons
    placemarks = list(folder_elem.findall(f'{ns}Placemark')) + list(folder_elem.findall('Placemark'))
    polygon_placemarks = [pm for pm in placemarks if pm.find(f'{ns}Polygon') is not None or pm.find('Polygon') is not None]
    
    # If less than 2 polygons, nothing to remove
    if len(polygon_placemarks) < 2:
        return 0
    
    # Remove first and last
    folder_elem.remove(polygon_placemarks[0])
    folder_elem.remove(polygon_placemarks[-1])
    
    return 2

def remove_unwanted_elements(root):
    """Remove unwanted folders and overlays by name from the KML"""
    unwanted_names = {
        'HYSPLIT Information',
        'NOAA',
        'NOAA NWS kml Weather Data',
        'NOAA NESDIS kml Smoke/Fire Data',
        'EPA AIRNow Air Quality Index (AQI)',
        'Soure Locations'  # typo in the actual file
    }
    
    removed_count = 0
    ns = '{http://www.opengis.net/kml/2.2}'
    
    # Find the Document
    doc = root.find(f'{ns}Document') or root.find('Document')
    if doc is None:
        print(f"    DEBUG: No Document found")
        return 0
    
    print(f"    DEBUG: Found Document, checking children...")
    
    # Remove ScreenOverlays and Folders that match unwanted names
    items_to_remove = []
    
    for child in list(doc):
        print(f"    DEBUG: Checking child: {child.tag}")
        
        # Check ScreenOverlays
        if 'ScreenOverlay' in child.tag:
            name_elem = child.find(f'{ns}name')
            print(f"      ScreenOverlay - name_elem: {name_elem}")
            if name_elem is not None:
                fname = name_elem.text.strip() if name_elem.text else ""
                print(f"      ScreenOverlay found: {repr(fname)}, in unwanted? {fname in unwanted_names}")
                if fname in unwanted_names:
                    items_to_remove.append((child, fname))
        
        # Check Folders
        elif 'Folder' in child.tag:
            name_elem = child.find(f'{ns}name')
            print(f"      Folder - name_elem: {name_elem}")
            if name_elem is not None:
                fname = name_elem.text.strip() if name_elem.text else ""
                fname = fname.replace('<pre>', '').replace('</pre>', '').strip()
                print(f"      Folder found: {repr(fname)}, in unwanted? {fname in unwanted_names}")
                if fname in unwanted_names:
                    items_to_remove.append((child, fname))
    
    print(f"    DEBUG: Items to remove: {len(items_to_remove)}")
    
    # Remove items
    for item, fname in items_to_remove:
        doc.remove(item)
        removed_count += 1
        print(f"    ✓ Removed: {fname}")
    
    return removed_count

def process_kmz(input_path, output_path):
    """Process a single KMZ file"""
    try:
        # Extract KMZ (it's a ZIP file)
        extract_dir = "./temp_kml"
        with zipfile.ZipFile(input_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        # Find and parse the KML file
        kml_file = None
        for file in os.listdir(extract_dir):
            if file.endswith('.kml'):
                kml_file = os.path.join(extract_dir, file)
                break
        
        if kml_file is None:
            print(f"  ❌ No KML file found in {input_path}")
            return False
        
        # Parse KML
        tree = ET.parse(kml_file)
        root = tree.getroot()
        
        # Process each folder
        ns = '{http://www.opengis.net/kml/2.2}'
        all_folders = root.findall(f'.//{ns}Folder')
        if not all_folders:
            all_folders = root.findall('.//Folder')
        
        print(f"  Found {len(all_folders)} folders")
        
        removed_from_folders = 0
        for i, folder in enumerate(all_folders):
            removed = remove_polygons_from_folder(folder)
            if removed > 0:
                folder_name = folder.find(f'{ns}name') or folder.find('name')
                fname = folder_name.text if folder_name is not None else f"Folder {i}"
                print(f"    {fname}: removed first and last polygons")
                removed_from_folders += 1
        
        # Remove any unwanted elements globally
        removed = remove_unwanted_elements(root)
        if removed > 0:
            print(f"  Removed {removed} unwanted element(s)")
        
        # Save modified KML
        tree.write(kml_file, encoding='utf-8', xml_declaration=True)
        
        # Re-package as KMZ
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as kmz:
            for file in os.listdir(extract_dir):
                file_path = os.path.join(extract_dir, file)
                kmz.write(file_path, arcname=file)
        
        # Cleanup temp directory
        shutil.rmtree(extract_dir)
        
        print(f"  ✓ Saved to {output_path}")
        return True
        
    except Exception as e:
        print(f"  ❌ Error processing {input_path}: {e}")
        return False

def main():
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Get all KMZ files
    kmz_files = list(Path(INPUT_DIR).glob('*.kmz'))
    
    if not kmz_files:
        print(f"No KMZ files found in {INPUT_DIR}")
        return
    
    print(f"Processing {len(kmz_files)} KMZ files...\n")
    
    successful = 0
    for kmz_path in kmz_files:
        print(f"Processing: {kmz_path.name}")
        output_path = os.path.join(OUTPUT_DIR, kmz_path.name)
        if process_kmz(str(kmz_path), output_path):
            successful += 1
        print()
    
    print(f"\n✓ Completed: {successful}/{len(kmz_files)} files processed successfully")

if __name__ == "__main__":
    main()

Processing 24 KMZ files...

Processing: jul24_los_cumpas_pm10.kmz
  Found 28 folders
    DEBUG: Found Document, checking children...
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}name
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}open
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}LookAt
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking child: {http://www.opengis.net/kml/2.2}Style
    DEBUG: Checking chi